# Churn Prediction Modeling

In the EDA notebook, churn appeared to be tied more to user consistency and account age than to device type or raw activity volume.

In this notebook, I test whether those behavior patterns are strong enough to help predict churn. I compare a simple logistic regression model with a random forest model, then focus on what the results mean for retention targeting.

The goal is to understand whether the data can help prioritize users who may need onboarding support, re-engagement, or habit-building nudges.

## 1. Load Cleaned Data

In [40]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Use the cleaned dataset from the EDA notebook so this step starts from documented assumptions.
df = pd.read_csv("/content/waze_churn_cleaned.csv")

df.head()

,id,label,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,...,churned,sessions_per_activity_day,drives_per_driving_day,km_per_drive,minutes_per_drive,fav_navigation_total,activity_rate,driving_rate,tenure_group,activity_group
0,0,retained,283,226,296.748273,2276,208,0,2628.845068,1985.775061,...,0.0,10.107143,11.894737,11.632058,8.786615,208,0.012302,0.008348,late_mid_tenure,highest_activity
1,1,retained,133,107,326.896596,1225,19,64,13715.920550,3160.472914,...,0.0,10.230769,9.727273,128.186173,29.537130,83,0.010612,0.008980,early_mid_tenure,low_mid_activity
2,2,retained,114,95,135.522926,2651,0,0,3059.148818,1610.735904,...,0.0,8.142857,11.875000,32.201567,16.955115,0,0.005281,0.003018,longest_tenure,low_mid_activity
3,3,retained,49,40,67.589221,15,322,7,913.591123,587.196542,...,0.0,7.000000,13.333333,22.839778,14.679914,329,0.466667,0.200000,newest_users,lowest_activity
4,4,retained,84,68,168.247020,1562,166,5,3950.202008,1219.555924,...,0.0,3.111111,3.777778,58.091206,17.934646,171,0.017286,0.011524,early_mid_tenure,highest_activity


In [41]:
# Confirm the file loaded correctly before modeling.
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nChurn distribution:")
print(df["churned"].value_counts(normalize=True).round(4))

df.info()

Rows: 14299
Columns: 23

Churn distribution:
churned
0.0    0.8226
1.0    0.1774
Name: proportion, dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14299 entries, 0 to 14298
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         14299 non-null  int64  
 1   label                      14299 non-null  object 
 2   sessions                   14299 non-null  int64  
 3   drives                     14299 non-null  int64  
 4   total_sessions             14299 non-null  float64
 5   n_days_after_onboarding    14299 non-null  int64  
 6   total_navigations_fav1     14299 non-null  int64  
 7   total_navigations_fav2     14299 non-null  int64  
 8   driven_km_drives           14299 non-null  float64
 9   duration_minutes_drives    14299 non-null  float64
 10  activity_days              14299 non-null  int64  
 11  driving_days               14299 non-null  i

## 2. Select Features

I’m using behavior, tenure, navigation, and device fields that could reasonably be known before churn happens. I’m leaving out the user ID, the original text label, and the numeric churn flag because those either identify the user or directly reveal the target.

In [42]:
target = "churned"

drop_cols = ["id", "label", "churned"]

X = df.drop(columns=drop_cols)
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['sessions', 'drives', 'total_sessions', 'n_days_after_onboarding', 'total_navigations_fav1', 'total_navigations_fav2', 'driven_km_drives', 'duration_minutes_drives', 'activity_days', 'driving_days', 'sessions_per_activity_day', 'drives_per_driving_day', 'km_per_drive', 'minutes_per_drive', 'fav_navigation_total', 'activity_rate', 'driving_rate']

Categorical features:
['device', 'tenure_group', 'activity_group']


## 3. Train/Test Split

Churned users make up a smaller share of the dataset, so I’m using a stratified split to keep the churn rate similar in both the training and test samples.

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Train churn rate:", round(y_train.mean(), 4))
print("Test churn rate:", round(y_test.mean(), 4))

Training rows: 10724
Testing rows: 3575
Train churn rate: 0.1774
Test churn rate: 0.1773


## 4. Preprocessing

Numeric fields are imputed with the median. Device type is one-hot encoded so it can be used by the models.

In [44]:
# Median imputation is a practical choice for the engineered ratio features.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Device is categorical, so one-hot encoding keeps it usable without implying order.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Keeping preprocessing inside the pipeline helps avoid train/test mismatch.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 4. Preprocessing

The dataset is mostly numeric, with device type as the main categorical feature. I’m using median imputation for numeric fields and one-hot encoding for device type.

In [45]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 5. Evaluation Function

I'm comparing models with precision, recall, F1 score, ROC-AUC, and the confusion matrix.

Recall matters here because it tells me how many actual churned users the model catches. Precision matters because it tells me how noisy the flagged churn-risk group would be.

In [46]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    print(model_name)
    print("-" * len(model_name))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return metrics

## 6. Logistic Regression Baseline

I’m starting with logistic regression because it is simple, explainable, and appropriate for a first churn model. I’m using class weighting because churned users are the minority group.

In [47]:
log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

log_reg_model.fit(X_train, y_train)

log_reg_metrics = evaluate_model(
    log_reg_model,
    X_test,
    y_test,
    "Logistic Regression"
)

Logistic Regression
-------------------
Confusion Matrix:
[[1989  952]
 [ 203  431]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.91      0.68      0.77      2941
         1.0       0.31      0.68      0.43       634

    accuracy                           0.68      3575
   macro avg       0.61      0.68      0.60      3575
weighted avg       0.80      0.68      0.71      3575



## 7. Random Forest Comparison

I’m also testing a random forest to see whether a more flexible model picks up patterns that logistic regression misses. I’m keeping the settings fairly simple so the model stays understandable for a portfolio project.

In [48]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=10
    ))
])

rf_model.fit(X_train, y_train)

rf_metrics = evaluate_model(
    rf_model,
    X_test,
    y_test,
    "Random Forest"
)

Random Forest
-------------
Confusion Matrix:
[[2376  565]
 [ 337  297]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.81      0.84      2941
         1.0       0.34      0.47      0.40       634

    accuracy                           0.75      3575
   macro avg       0.61      0.64      0.62      3575
weighted avg       0.78      0.75      0.76      3575



## 8. Compare Model Results

In [49]:
model_results = pd.DataFrame([log_reg_metrics, rf_metrics])

metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
model_results[metric_cols] = model_results[metric_cols].round(4)

model_results

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.6769,0.3116,0.6798,0.4274,0.7406
1,Random Forest,0.7477,0.3445,0.4685,0.3971,0.7289


## 9. Interpret the Tradeoff

The model comparison is less about picking the “best” score and more about understanding the tradeoff.

For retention work, a model with higher recall may be more useful because it catches more users who actually churned. A model with higher precision would create a cleaner outreach list, but may miss more churned users.

In [50]:
best_recall_model = model_results.sort_values("recall", ascending=False).iloc[0]
best_auc_model = model_results.sort_values("roc_auc", ascending=False).iloc[0]

print("Best recall model:", best_recall_model["model"])
print("Best ROC-AUC model:", best_auc_model["model"])

Best recall model: Logistic Regression
Best ROC-AUC model: Logistic Regression


## 10. Review Top Random Forest Signals

I’m using random forest feature importance as a directional check. This does not prove causality, but it helps identify which fields the model relied on most.

In [51]:
# Get readable feature names after preprocessing.
fitted_preprocessor = rf_model.named_steps["preprocessor"]

encoded_device_features = (
    fitted_preprocessor
    .named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(categorical_features)
    .tolist()
)

feature_names = numeric_features + encoded_device_features

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

top_features = feature_importance.head(12)

top_features

,feature,importance
8,activity_days,0.115568
9,driving_days,0.094142
3,n_days_after_onboarding,0.066763
15,activity_rate,0.055808
16,driving_rate,0.054890
10,sessions_per_activity_day,0.052730
14,fav_navigation_total,0.044782
7,duration_minutes_drives,0.044064
11,drives_per_driving_day,0.042379
4,total_navigations_fav1,0.042118


## 11. Create Churn Scores for Dashboard Exploration

To support dashboarding, I’m scoring users with predicted churn probability. I would not treat these scores as final decisions. They are a way to explore which types of users appear higher-risk.

In [52]:
# Use the random forest probabilities for dashboard exploration because it had the stronger overall ranking score.
df_scored = df.copy()

X_all = df_scored.drop(columns=drop_cols)

df_scored["predicted_churn_probability"] = rf_model.predict_proba(X_all)[:, 1]

df_scored[[
    "id",
    "churned",
    "predicted_churn_probability",
    "activity_days",
    "driving_days",
    "n_days_after_onboarding",
    "sessions",
    "drives",
    "device"
]].head()

,id,churned,predicted_churn_probability,activity_days,driving_days,n_days_after_onboarding,sessions,drives,device
0,0,0.0,0.047212,28,19,2276,283,226,Android
1,1,0.0,0.417857,13,11,1225,133,107,iPhone
2,2,0.0,0.185841,14,8,2651,114,95,Android
3,3,0.0,0.587343,7,3,15,49,40,iPhone
4,4,0.0,0.027507,27,18,1562,84,68,Android


## 12. Create Simple Risk Groups

For the dashboard, I’m grouping predicted churn probabilities into four equal-sized risk groups. This makes it easier to compare user behavior across lower-risk and higher-risk segments.

In [53]:
df_scored["risk_group"] = pd.qcut(
    df_scored["predicted_churn_probability"],
    q=4,
    labels=["Lowest Risk", "Lower-Mid Risk", "Upper-Mid Risk", "Highest Risk"]
)

risk_summary = (
    df_scored.groupby("risk_group", observed=True)
    .agg(
        users=("id", "count"),
        actual_churn_rate=("churned", "mean"),
        avg_predicted_churn_probability=("predicted_churn_probability", "mean"),
        avg_activity_days=("activity_days", "mean"),
        avg_driving_days=("driving_days", "mean"),
        avg_tenure_days=("n_days_after_onboarding", "mean"),
        avg_sessions=("sessions", "mean"),
        avg_drives=("drives", "mean")
    )
    .reset_index()
)

for col in ["actual_churn_rate", "avg_predicted_churn_probability"]:
    risk_summary[col] = risk_summary[col].round(4)

risk_summary

,risk_group,users,actual_churn_rate,avg_predicted_churn_probability,avg_activity_days,avg_driving_days,avg_tenure_days,avg_sessions,avg_drives
0,Lowest Risk,3575,0.0129,0.0859,24.893147,20.149371,2189.434965,72.379021,60.558042
1,Lower-Mid Risk,3575,0.0257,0.2066,19.101818,15.121399,1803.417343,76.593287,63.960559
2,Upper-Mid Risk,3574,0.1416,0.3845,12.353945,9.386961,1671.344712,83.162843,69.271964
3,Highest Risk,3575,0.5292,0.6452,5.828811,4.071608,1343.070490,90.360839,75.233287


## 13. Export Modeling Outputs

In [54]:
df_scored.to_csv("waze_churn_scored.csv", index=False)
model_results.to_csv("model_results.csv", index=False)
feature_importance.to_csv("feature_importance.csv", index=False)
risk_summary.to_csv("risk_summary.csv", index=False)

print("Exported files:")
print("- waze_churn_scored.csv")
print("- model_results.csv")
print("- feature_importance.csv")
print("- risk_summary.csv")

Exported files:
- waze_churn_scored.csv
- model_results.csv
- feature_importance.csv
- risk_summary.csv


## Key Modeling Takeaways

The model results support the EDA finding that churn is tied more to consistency and tenure than raw activity volume.

Logistic regression had stronger recall, meaning it identified a larger share of users who actually churned. Random forest had slightly stronger overall accuracy and ROC-AUC, making it useful for ranking users by predicted churn risk.

The random forest feature importance also pointed to activity days, driving days, account age, activity rate, and driving rate as the strongest churn signals. Device type did not appear to be a major driver.

The risk-group table gives the clearest business takeaway. Users in the highest-risk segment had a 52.9% actual churn rate, compared with only 1.3% in the lowest-risk segment. The highest-risk users also had fewer active days, fewer driving days, and shorter tenure, even though they averaged more sessions and drives.

This suggests that churn risk is less about total usage volume and more about whether users develop consistent app habits over time.